Query Preprocessing:
The purpose of this component is to prepare user questions before they are sent to the retrieval system. It validates the input, performs basic text cleaning, and converts the question into a numerical vector (embedding) using the same embedding model used to represent the product documents.

Workflow:
The user submits a question.
The question is validated and cleaned.
The cleaned question is converted into an embedding.
The resulting embedding is passed to the retriever.
The retriever searches the ChromaDB collection for emantically relevant product documents.

In [1]:
# Importing Required Libraries
import numpy as np

In [2]:
#Defining the Query Preprocessing Function
def preprocess_query(query):
    """
    Validate and clean a user query.

    Parameters:
        query (str): The original question submitted by the user.

    Returns:
        str: The cleaned query.
    """

    # Check that the input is a string
    if not isinstance(query, str):
        raise TypeError("Query must be a string.")

    # Remove unnecessary whitespace
    query = query.strip()

    # Reject empty queries
    if not query:
        raise ValueError("Query cannot be empty.")

    return query

In [3]:
#Testing Query Preprocessing 
# Test 1: Valid query with extra whitespace
query = "   Which products contain probiotics?   "
cleaned_query = preprocess_query(query)

print("Original query:", repr(query))
print("Cleaned query:", repr(cleaned_query))


Original query: '   Which products contain probiotics?   '
Cleaned query: 'Which products contain probiotics?'


In [4]:
#Testing Invalid Queries
# Test 2: Empty query
try:
    preprocess_query("")
except ValueError as e:
    print("Empty query:", e)


# Test 3: Query containing only whitespace
try:
    preprocess_query("     ")
except ValueError as e:
    print("Whitespace query:", e)


# Test 4: Invalid input type
try:
    preprocess_query(123)
except TypeError as e:
    print("Invalid input type:", e)
    

Empty query: Query cannot be empty.
Whitespace query: Query cannot be empty.
Invalid input type: Query must be a string.


In [5]:
import os

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\admin\Desktop\RAG-\rag


In [6]:
from embeddings import get_embedding

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded!
Embedding dimensions: 384


c:\Users\admin\Desktop\RAG-\rag\embeddings.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


In [7]:
#Generating Query Embeddings
# Example user query
query = "   Which products contain probiotics?   "

# Preprocess the query
cleaned_query = preprocess_query(query)

# Generate its embedding
query_embedding = get_embedding(cleaned_query)

print("Cleaned query:", cleaned_query)
print("Embedding dimensions:", len(query_embedding))
print("First 5 values:", query_embedding[:5])

Cleaned query: Which products contain probiotics?
Embedding dimensions: 384
First 5 values: [-0.04810575023293495, -0.09583279490470886, -0.015988826751708984, -0.01868784986436367, 0.04856337979435921]


In [8]:
assert len(query_embedding) == 384, "Incorrect embedding dimensions!"

print("Dimension check passed:", len(query_embedding))


Dimension check passed: 384


In [9]:
assert all(isinstance(value, (int, float)) for value in query_embedding), \
    "Embedding contains non-numerical values!"

print("All embedding values are numerical.")


All embedding values are numerical.


In [10]:
embedding_norm = np.linalg.norm(query_embedding)

print("Embedding norm:", embedding_norm)

assert np.isclose(embedding_norm, 1.0, atol=1e-5), \
    "Embedding is not normalized!"

print("Normalization check passed.")

Embedding norm: 1.0000000349690326
Normalization check passed.


In [11]:
#Creating the structured output
def prepare_query_for_retrieval(query):
    """
    Validate, clean, and embed a user query.

    Parameters:
        query (str): Original user question.

    Returns:
        dict: Cleaned query and its embedding.
    """

    # Step 1: Validate and clean the query
    cleaned_query = preprocess_query(query)

    # Step 2: Generate its embedding
    query_embedding = get_embedding(cleaned_query)

    # Step 3: Organize the results
    prepared_query = {
        "query": cleaned_query,
        "embedding": query_embedding
    }

    return prepared_query

In [12]:
# Testing the complete function
# Example user question
user_query = "   Do you have vitamin C supplements?   "

# Prepare the query
prepared_query = prepare_query_for_retrieval(user_query)

# Display the results
print("Prepared query:", prepared_query["query"])
print("Embedding dimensions:", len(prepared_query["embedding"]))
print("Embedding norm:", np.linalg.norm(prepared_query["embedding"]))

Prepared query: Do you have vitamin C supplements?
Embedding dimensions: 384
Embedding norm: 0.999999932467737


In [13]:
#Final Verification
# Verify output structure
assert isinstance(prepared_query, dict)

assert "query" in prepared_query
assert "embedding" in prepared_query

# Verify query
assert isinstance(prepared_query["query"], str)
assert prepared_query["query"] != ""

# Verify embedding
assert len(prepared_query["embedding"]) == 384

assert all(
    isinstance(value, (int, float))
    for value in prepared_query["embedding"]
)

assert np.isclose(
    np.linalg.norm(prepared_query["embedding"]),
    1.0,
    atol=1e-5
)

print("All final verification checks passed!")
print("Query preprocessing component is ready for integration.")

All final verification checks passed!
Query preprocessing component is ready for integration.


##  Conclusion

In this notebook, we implemented the query preprocessing component of the Pharmaly RAG chatbot.

The component performs the following operations:

1. Validates incoming user questions.
2. Removes unnecessary whitespace.
3. Generates query embeddings using the existing `paraphrase-multilingual-MiniLM-L12-v2` model.
4. Validates embedding dimensions, numerical values, and normalization.
5. Returns a structured output containing the cleaned question and its embedding.

The component reuses the embedding implementation developed by Member 2, ensuring consistency between query embeddings and the product document embeddings stored in ChromaDB.

### Output

The `prepare_query_for_retrieval()` function returns a dictionary containing:

- `query`: the cleaned user question.
- `embedding`: its normalized 384-dimensional vector.

This output is ready to be consumed by the retrieval component.

The next stage of the RAG pipeline will use this embedding to retrieve semantically relevant Pharmaly product documents from ChromaDB.